# Data Sourcing

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/01_data_sourcing.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️
```

## Setup

### Housekeeping (no interaction required)

In [ ]:
%pip install impresso

❗ Please restart the kernel/runtime after installing the package to ensure that all changes take effect.

(Google Colab might initiate a restart on its own)

In [ ]:
import os
import random
import time
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [ ]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Ask for a yes/no confirmation in the notebook.

    Args:
        question: Prompt shown to the user.

    Returns:
        True for yes, False for no.
    """
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

<img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=12> The cell below will attempt to connect to your Google Drive.

*Once prompted, give all demanded permissions*

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

### Setup (interaction required)

#### Connect with the impresso API

You can generate an API Access Token in [Impresso Datalab](https://impresso-project.ch/datalab/token/).

The cell below will prompt you for this token.

In [ ]:
import impresso

client = impresso.connect()

## Explore impresso

Play around in the [Impresso App](https://impresso-project.ch/app) and try to find a query that creates a small and interesting corpus of around 4000 articles.

Depending on your interest, try to create a query that returns data that is temporally dispersed.

![temporal-dispersion](../assets/impresso_temporal_dispersion.png)

This can mean that you need to combine different spellings or old terms in your query, for example adding *Armuth* to an *Armut*-query.

Once you have found an interesting query, you can click "Try in Datalab" to extract the search terms and enter them below.

![try-in-datalab](../assets/try_in_datalab.png)


## Create your corpus

Impresso can return different types of items where search terms appear. Most common are 'pages' and 'articles'. Pages contain the text of a whole page where a term occurs, while the articles are limited to the article.

In consequence, this means that units of type 'page' can both contain a lot of text unrelated to the search terms, as well as not all text related to the search term, if a relevant article spans more than one page.

You can restrict your corpus to only articles by setting the value below.

❗ You cannot directly paste the query down below. You need to turn the parameter names into strings and make a dictionary out of them, like so:

| Impresso Suggestion                        | Translation to paste below                        |
| ------------------------------------------ | ------------------------------------------------- |
| ![bool](../assets/withtextcontents.png)    | `"with_text_contents": True,`                     |
| ![single value](../assets/singlevalue.png) | `"country": "CH",`                                |
| ![orvalue](../assets/orvalue.png)          | `"partner_id": impresso.OR(["SNL", "Migros")],`   |
| ![daterange](../assets/daterange.png)      | `"date_range": impresso.DateRange("1880-01-01", "1970-12-31"),` |



In [ ]:
# ⬇️⬇️⬇️ Your search query here
QUERY_PARAMS = {
    "with_text_contents": True,
    "language": "de",
    "term": "Armenpflege",
}

ONLY_ARTICLES = False
# ⬆️⬆️⬆️

In [ ]:
# ⬇️⬇️⬇️  This name will be used to save your files and throughout the rest of the summer school.
CORPUS_NAME = "armenpflege"
# ⬆️⬆️⬆️ 

### Create an index

First, we create an index of all articles defined by your query.

The index will not contain the fulltext of the articles, but it will contain the *"id"* of each article.
This complete list of ids will be used in the next step to download the fulltext.

In [ ]:
search_results = client.search.find(
	**QUERY_PARAMS,
)

print(f"Your search query returned {search_results.total} results!")
if ONLY_ARTICLES:
    print("This number will likely be smaller once articles are filtered out.")
print()
if search_results.total > 5000:
    print("⚠️ Warning: Your search query returns more than 5000 results!")
    print("The download will take a long time, and not all modules of the summer school may be feasible to execute.")
    
search_results

In [ ]:
INDEXPATH = DATA_DIR / f"{CORPUS_NAME}.index.csv"

In [ ]:
TOTAL_RESULTS = search_results.total
LIMIT = 250 # Maximum is 1000
MAX_RETRIES = 5

all_results = []
n_retrieved = 0
n_kept = 0

for offset in range(0, TOTAL_RESULTS, LIMIT):

    if ONLY_ARTICLES:
        print(f"\rRetrieved {n_retrieved} / {TOTAL_RESULTS} items so far (keeping {n_kept} Articles)...", end="")
    else:
        print(f"\rRetrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="")

    for attempt in range(MAX_RETRIES):
        try:
            search_results = client.search.find(
                **QUERY_PARAMS,
                limit=LIMIT,
                offset=offset
            )
            break
        except Exception as e:
            if attempt < MAX_RETRIES:
                print(f"\nConnection error at {offset}. Reconnecting... (Attempt {attempt + 1}/{MAX_RETRIES})")
                time.sleep(5)
                client = impresso.connect() # This automates your manual fix
            else:
                print(f"\nFinal attempt failed at {offset}.")
                raise e

    if ONLY_ARTICLES:
        df = search_results.df[search_results.df["text.itemTypeLabel"] == "Article"]
    else:
        df = search_results.df

    all_results.append(df)
    n_retrieved += len(search_results.df)
    n_kept += len(df)

    if ONLY_ARTICLES:
        print(f"\rRetrieved {n_retrieved} / {TOTAL_RESULTS} items so far (keeping {n_kept} Articles)...", end="")
    else:
        print(f"\rRetrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="")

index_df: pd.DataFrame = pd.concat(all_results)

# Save only unique index values, for resuming after interruptions.
index_df = index_df[~index_df.index.duplicated(keep='first')]
index_df.to_csv(INDEXPATH, index=True)

### Download the full text for all found articles

The cell below determines how many articles still need to be retrieved.

That way, if the download is interrupted at any point, we don't need to re-download everything that has already been saved.

In [ ]:
TIMEOUT = 2 # Be nice to the server and avoid sending too many requests in a short time
OUTPATH = DATA_DIR / f"{CORPUS_NAME}.wide.parquet"

def load_missing_ids(print_info: bool = True) -> tuple[list[str], pd.DataFrame]:
    """Return IDs that still need download and the already downloaded rows.

    Args:
        print_info: Whether to print a short progress estimate.

    Returns:
        A tuple with:
            - Missing document IDs.
            - Existing fulltext dataframe (empty if no file exists yet).
    """
    try:
        index_df = pd.read_csv(INDEXPATH, index_col="id")
        wanted_ids = set(index_df.index.to_list())
    except FileNotFoundError:
        wanted_ids = set()

    try:
        fulltext_df = pd.read_parquet(OUTPATH)
        retrieved_ids = set(fulltext_df.index.to_list())
    except FileNotFoundError:
        fulltext_df = pd.DataFrame()
        retrieved_ids = set()

    missing_ids = [doc_id for doc_id in wanted_ids if doc_id not in retrieved_ids]

    if print_info:
        print(f"Missing {len(missing_ids)} items. (Already retrieved {len(retrieved_ids)})")
        print(f"Requests for missing items will take approximately {len(missing_ids) * TIMEOUT / 60:.2f} minutes (assuming ~{1 / TIMEOUT:.2f} request per second).")

    return missing_ids, fulltext_df

missing_ids, fulltext_df = load_missing_ids()

Retrieve all missing documents.

If an error occurs, you can just restart the cell.
The progress is continuously saved and the cell will pick up the work where it left it.

In [ ]:
MAX_RETRIES = 5

# To avoid accidental overwrites, this cell loads the data again in the beginning
missing_ids, fulltext_df = load_missing_ids()

try:
    for i, doc_id in enumerate(missing_ids):
        time.sleep(random.uniform(TIMEOUT * 0.5, TIMEOUT * 1.5))
        print(f"\r({i+1}/{len(missing_ids)}) - Processing document {doc_id}", end="")
        for attempt in range(MAX_RETRIES):
            try:
                content_item = client.content_items.get(doc_id)
                content_item_df = content_item.df
                content_item_df = content_item_df.set_index("id")
                fulltext_df = pd.concat([fulltext_df, content_item_df], ignore_index=False)
                break

            except Exception as e:
                if attempt < MAX_RETRIES:
                    print(f"\nConnection error on {doc_id}. Reconnecting... (Retry {attempt+1}/{MAX_RETRIES})")
                    time.sleep(5)
                    client = impresso.connect()
                else:
                    print(f"\nFinal attempt failed on {doc_id}.")
                    raise e

        if (i + 1) % 100 == 0: # Save progress every 100 items
            fulltext_df.to_parquet(OUTPATH, index=True)
    print() # Move to the next line after the loop finishes

except KeyboardInterrupt:
    print("Process interrupted by user. Saving progress...                              ")
    fulltext_df.to_parquet(OUTPATH, index=True)

except Exception as e:
    print("An error occurred. Saving progress...                              ")
    fulltext_df.to_parquet(OUTPATH, index=True)
    raise e

fulltext_df.to_parquet(OUTPATH, index=True)

### Save the corpus with fewer columns

Saving the corpus with only the needed columns saves loading times and memory down the line.
Furthermore, we rename the columns to a more practical naming scheme.

For the rest of the summer school, we will only be working with the smaller version created below.

In [ ]:
reduced_df = fulltext_df[["meta.date", "meta.countryCode", "meta.provinceCode", "meta.mediaTitle", "text.langCode", "text.content"]]

reduced_df = reduced_df.rename(columns={
    "meta.date": "date",
    "meta.countryCode": "country_code",
    "meta.provinceCode": "province_code",
    "meta.mediaTitle": "media_title",
    "text.langCode": "lang_code",
    "text.content": "content",
})

# reduced_df.to_csv(os.path.join(DATA_DIR, f"{CORPUS_NAME}.raw.csv"), index=True)
reduced_df.to_parquet(os.path.join(DATA_DIR, f"{CORPUS_NAME}.full.parquet"), index=True)